# AutoGluon — End-to-End ML with Metrics
### Part 3

A complete, realistic ML workflow with AutoGluon on a single dataset: from raw data to a tuned
model with a **full metrics report** and diagnostic plots. Where Part 2 toured breadth, this part
goes **deep on one problem** and the evaluation story.

**Pipeline:**
1. Load & explore data (Titanic survival — binary classification)
2. Train/validation/test split
3. `TabularPredictor.fit()` with `best_quality`
4. Leaderboard & model internals (`fit_summary`)
5. Full metrics: accuracy, precision, recall, F1, ROC-AUC, log-loss
6. Confusion matrix + ROC curve + precision-recall curve
7. Feature importance
8. Save / load the trained predictor (deployment)

> Runtime: CPU is fine. `Runtime → Run all`.

In [ ]:
!pip install -q autogluon.tabular[all] scikit-learn matplotlib seaborn
print("Installed. Restart runtime if prompted, then Run all.")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from autogluon.tabular import TabularPredictor
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
                             precision_recall_curve, classification_report)
sns.set_style("whitegrid")

## 1. Data — Titanic survival

In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
# light cleanup; AutoGluon handles the rest (missing values, encoding)
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
label = "Survived"
print(df.shape)
df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df[label].value_counts().plot(kind="bar", ax=ax[0], title="Class balance (0=died,1=survived)")
sns.histplot(data=df, x="Age", hue=label, bins=30, ax=ax[1])
ax[1].set_title("Age distribution by survival")
plt.show()

## 2. Split into train / test

We hold out 20% as a true test set the model never sees during training.

In [ ]:
test = df.sample(frac=0.2, random_state=42)
train = df.drop(test.index)
print("train:", train.shape, " test:", test.shape)

## 3. Train with best_quality

`best_quality` enables bagging + multi-layer stacking for the strongest ensemble.

In [ ]:
predictor = TabularPredictor(label=label, eval_metric="roc_auc").fit(
    train, time_limit=180, presets="best_quality")

## 4. Leaderboard & fit summary

In [ ]:
predictor.leaderboard(test, silent=True)

In [ ]:
summary = predictor.fit_summary(verbosity=1)

## 5. Full metrics report

AutoGluon's `evaluate` reports several metrics at once; we add a sklearn classification report.

In [ ]:
X_test = test.drop(columns=[label])
y_test = test[label].values

print("=== AutoGluon evaluate ===")
metrics = predictor.evaluate(test, detailed_report=False)
for k, v in metrics.items():
    print(f"  {k:<12}: {v:.4f}")

print("\n=== sklearn classification report ===")
y_pred = predictor.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

## 6. Diagnostic plots — confusion matrix, ROC, PR curve

In [ ]:
y_proba = predictor.predict_proba(X_test).iloc[:, 1].values

fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[0])
ax[0].set_title("Confusion matrix"); ax[0].set_xlabel("predicted"); ax[0].set_ylabel("actual")

# ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax[1].plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.3f}")
ax[1].plot([0, 1], [0, 1], "k--")
ax[1].set_title("ROC curve"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].legend()

# Precision-Recall
prec, rec, _ = precision_recall_curve(y_test, y_proba)
ax[2].plot(rec, prec)
ax[2].set_title("Precision-Recall curve"); ax[2].set_xlabel("recall"); ax[2].set_ylabel("precision")
plt.show()

## 7. Feature importance

In [ ]:
fi = predictor.feature_importance(test)
fi

In [ ]:
fi["importance"].sort_values().plot(kind="barh", figsize=(7,5),
                                     title="Permutation feature importance")
plt.tight_layout(); plt.show()

## 8. Save & load (deployment)

AutoGluon persists the whole pipeline to disk. Reload later for inference with no retraining.

In [ ]:
path = predictor.path
print("Model saved at:", path)

reloaded = TabularPredictor.load(path)
sample = X_test.head(5)
print("Predictions from reloaded model:")
print(reloaded.predict(sample).tolist())
print("Probabilities:")
print(reloaded.predict_proba(sample).round(3))

## Summary

A full AutoGluon workflow end to end:
- Minimal cleaning — AutoGluon handles missing values, encoding, scaling internally.
- `best_quality` built a stacked ensemble; the leaderboard showed every model.
- We reported **accuracy, precision, recall, F1, ROC-AUC, log-loss** and visualized the
  confusion matrix, ROC, and PR curves.
- Feature importance explained the drivers (sex, fare, class dominate Titanic survival).
- The predictor saves/loads as a deployable artifact.